# Teacher-student test (distillation)

Can a small checker learn more from a big one than from the human labels alone?

- **Teacher:** LettuceDetect-large (396M parameters, span F1 58.9 on the RAGTruth test set).
- **Students:** Ettin 17M (and later 68M), the same small encoders TinyLettuce is built on
  (TinyLettuce 17M: 44.3, 68M: 50.1).

The teacher gives the student two things:

1. **Soft labels** on RAGTruth's training answers: its probability for every word, next to the
   human label (half and half).
2. **New answers with the teacher's labels.** Small, phone-sized models (Qwen2.5 0.5B and 1.5B,
   SmolLM2 1.7B) write their own answers to RAGTruth's training prompts. No person labels them;
   the teacher does. Test prompts are never used.

Three students of each size, trained the same way (same start, settings and steps):

| Run | Learns from |
|---|---|
| `hard17m` | human labels only (the control) |
| `soft17m` | human labels + teacher's soft labels |
| `kd17m` | the above + the new answers labelled by the teacher |

| Step | Needs | Time on a T4 (rough estimate) |
|---|---|---|
| Setup | internet | 3 min |
| 1. Teacher labels the training answers | GPU | 15-25 min |
| 2. Small models write new answers (3 models) | GPU | 60-90 min |
| 3. Teacher labels the new answers | GPU | 10 min |
| 4. Train and score the three 17M students | GPU | 45-90 min |
| 5. Fairness check (20 min), then the 68M students if the gain holds | GPU | 2-3.5 hours |

Select **Runtime > Change runtime type > T4 GPU** and run the cells in order. Everything is saved to
`MyDrive/ondevice_hallu/work`, and finished parts are skipped, so after a disconnect run the setup
cells again and then the step that was running. (A student whose training was cut off starts that
training again.) It needs the size ladder's results in the same folder (the published models'
test predictions).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, subprocess, shutil
from pathlib import Path
from IPython.display import Markdown, display

REPO = 'https://github.com/gbirhanu/ondevice-hallu.git'
CODE = Path('/content/ondevice_hallu')
DRIVE = Path('/content/drive/MyDrive/ondevice_hallu')
WORK = DRIVE / 'work'                       # the size-ladder results live here
DATA = Path('/content/RAGTruth')

def sh(command):
    result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                            env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
    print(result.stdout[-3000:], flush=True)
    return result.returncode

if (CODE / '.git').exists():
    sh(['git', '-C', str(CODE), 'pull', '--ff-only'])
else:
    assert sh(['git', 'clone', REPO, str(CODE)]) == 0, f'Cannot clone {REPO}: push the code first.'
if not DATA.exists():
    sh(['git', 'clone', '--depth', '1', 'https://github.com/ParticleMedia/RAGTruth.git', str(DATA)])
sh([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.6', 'accelerate'])
need = ['results/models.json', 'preds/ld_large_test.json.gz', 'preds/tiny17m_test.json.gz', 'preds/tiny68m_test.json.gz']
missing = [n for n in need if not (WORK / n).exists()]
print('Size-ladder files missing:', missing or 'none (good)')

In [ ]:
def run(*arguments):
    """Runs one run.py command and shows its output as it comes. Returns True when it worked."""
    command = [sys.executable, '-u', str(CODE / 'run.py'), *map(str, arguments),
               '--data-dir', str(DATA), '--work-dir', str(WORK)]
    process = subprocess.Popen(command, cwd=CODE, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               env={**os.environ, 'TOKENIZERS_PARALLELISM': 'false'})
    for line in process.stdout:
        if 'Loading checkpoint shards' not in line and 'Warning' not in line and 'Writing model shards' not in line:
            print(line, end='', flush=True)
    return process.wait() == 0

def free_disk():
    """Delete downloaded model weights from Colab's disk (not from Drive)."""
    hub = Path(os.environ.get('HF_HOME', Path.home() / '.cache' / 'huggingface')) / 'hub'
    for d in hub.glob('models--*'):
        shutil.rmtree(d, ignore_errors=True)

import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
TEACHER = 'ld_large'
SMALL = ['qwen2.5-0.5b', 'qwen2.5-1.5b', 'smollm2-1.7b']      # phone-sized answering models

## Step 1: the teacher labels RAGTruth's training answers

In [ ]:
assert run('teach', '--name', TEACHER), 'The teacher step failed; see the message above.'

## Step 2: small models write new answers to the training prompts

One answer per training source (2,264 prompts) from each small model. Answers that never finish
(cut at 450 tokens) or are empty are left out.

In [ ]:
for g in SMALL:
    if not run('generate', '--generator', g):
        print(f'--> {g} did not finish; the students use whatever was written.')
    free_disk()

## Step 3: the teacher labels the new answers

In [ ]:
assert run('teach', '--name', TEACHER), 'The teacher step failed; see the message above.'
free_disk()

## Step 4: train and score the 17M students

Same start (`jhu-clsp/ettin-encoder-17m`), same settings for all three: 5 epochs, learning rate
2e-5, the epoch with the best dev span F1 kept (dev = 10% of the training sources, never used for
training). Then each is scored on the 2,700 test answers.

In [ ]:
def students(size, runs):
    # runs: [(kind, epochs)]; kind = hard (human labels), soft (+ teacher) or kd (+ new answers)
    extra = [g for g in SMALL if (WORK / 'results' / f'extra_{g}.json').exists()]
    print('New answers from:', extra or 'none (run step 2)')
    options = {'hard': [], 'soft': ['--teacher', TEACHER], 'kd': ['--teacher', TEACHER, '--extra', *extra]}
    for kind, epochs in runs:
        name = f'{kind}{size}' + ('' if epochs == 5 else f'-{epochs}ep')
        if kind == 'kd' and not extra:
            print(f'--> {name} skipped: no new answers yet.')
            continue
        common = ['--base', f'jhu-clsp/ettin-encoder-{size}', '--epochs', epochs, '--lr', 2e-5]
        if run('train', '--name', name, *common, *options[kind]):
            run('score', '--name', name)
        else:
            print(f'--> {name} failed; see the message above.')
    run('distill')
    display(Markdown((WORK / 'results' / 'distill.md').read_text()))

def still_wins(name):
    # True when the run beats its matched human-labels run, with both 95% intervals above zero
    rows = {r['name']: r for r in json.loads((WORK / 'results' / 'distill.json').read_text())['rows']}
    r = rows.get(name, {})
    ok = bool(r.get('control')) and r['vs_control_05_ci'][0] > 0 and r['vs_control_cut_ci'][0] > 0
    print(f"{name} vs {r.get('control')}: {r.get('vs_control_05')} {r.get('vs_control_05_ci')} at 0.5, "
          f"{r.get('vs_control_cut')} {r.get('vs_control_cut_ci')} at the dev cut-off -> "
          + ('STILL WINS' if ok else 'NO LONGER CLEAR'))
    return ok

students('17m', [('hard', 5), ('soft', 5), ('kd', 5)])

**How to read it.** Each student with the teacher is compared with the human-labels-only student
of the same size that saw about as many answers in training (column **Compared with**):

- **Change, 0.5** and **Change, dev cut-off**: the gain from the teacher, with its 95% interval.
  The dev cut-off is chosen on the dev answers for each run (a student that learned soft labels may
  need a lower cut-off than 0.5).
- **vs TinyLettuce, 0.5**: against the published model of the same size.
- `soft17m` alone may gain little: the teacher was trained on these same training answers, so its
  labels there are close to the human ones. The new answers in `kd17m` are where the teacher adds
  something new.

## Step 5: a fairness check, then the 68M students

`kd17m` saw about 1.5 times as many answers in training as `hard17m` (13,584 human-labelled +
6,719 new, 5 times each). So first a human-labels-only student is trained for 8 epochs, which
sees about as many (`hard17m-8ep`, about 20 minutes), and `kd17m` is compared with it. Only if
`kd17m` still wins (both intervals above zero) does the cell go on to the 68M students: human
labels for 8 epochs, and teacher + new answers for 5 (about 2-3 hours).

In [ ]:
students('17m', [('hard', 8)])
if still_wins('kd17m'):
    students('68m', [('hard', 8), ('kd', 5)])
    still_wins('kd68m')
else:
    print('Stopped before the 68M students: tell Claude.')